# Bài 24 — chuỗi luyện Easy · LeetCode #119 — Pascal's Triangle II

**Mức độ:** Easy · **Chủ đề:** Dynamic Programming · 1D In-place · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/pascals-triangle-ii/](https://leetcode.com/problems/pascals-triangle-ii/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


**Lộ trình đầy đủ:** đề → nền tảng → baseline → điểm mấu chốt → code → trace → chứng minh → nộp LeetCode → Big-O → lỗi → biến thể → 6 bài tập → gợi ý → đáp án → checklist.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho chỉ số hàng `rowIndex` bắt đầu từ 0, trả về **chỉ hàng đó** của tam giác Pascal. `rowIndex=0` tương ứng `[1]`.

### Ví dụ quan trọng

`rowIndex=3 → [1,3,3,1]`; `rowIndex=0 → [1]`; `rowIndex=1 → [1,1]`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import List, Optional
from collections import deque, Counter

### 2.1. `list` là gì? Thay đổi từng phần tử bằng chỉ số

Trong Python, `list` là dãy có thứ tự. Ví dụ `[10,20,30]` chứa ba phần tử ở chỉ số
`0`, `1`, `2`. Truy cập `a[i]` để đọc, gán `a[i] = value` để sửa phần tử tại vị trí i.
List là **mutable** (có thể thay đổi sau khi tạo). Vì vậy thuật toán tối ưu bộ nhớ có thể
tái sử dụng list hiện tại; nhưng sửa list quá sớm cũng có thể phá dữ liệu còn cần dùng.

`len(a)` trả số phần tử, còn chỉ số cuối là `len(a)-1`. Đọc `a[len(a)]` gây `IndexError`.
Hãy dự đoán giá trị cuối của list trước khi chạy thí nghiệm.

In [ ]:
list_demo=[10,20,30]
print('ban đầu:',list_demo,'độ dài:',len(list_demo))
print('vị trí 0:',list_demo[0],'vị trí cuối:',list_demo[-1])
list_demo[1]=99
print('sửa phần tử thứ hai:',list_demo)
assert list_demo==[10,99,30]

### 2.2. `for`, `range` và vì sao ranh giới phải chính xác

`for i in range(n)` duyệt các số nguyên `0,1,...,n-1`: *n không nằm trong range*.
`range(start, stop, step)` điều chỉnh điểm bắt đầu, điểm dừng và bước nhảy.
Nếu `step` âm, vòng lặp đi từ phải qua trái; đây là công cụ thiết yếu cho cập nhật
*in-place* (đặc biệt quan trọng ở Pascal II).

Luôn viết thử 3–4 giá trị chỉ số bằng tay trước khi thêm `a[i+1]` hoặc `a[i-1]`.
Việc chọn sai biên có thể gây truy cập ngoài mảng hoặc bỏ sót phần tử cuối.

In [ ]:
print('range(4):', list(range(4)))
print('range(1,4):', list(range(1,4)))
print('range(3,0,-1):', list(range(3,0,-1)))
a_indices=[5,8,13,21]
for i in range(1,len(a_indices)):
    print('cặp liên tiếp:',a_indices[i-1],a_indices[i])

### 2.3. `append`, phép cộng list và sao chép an toàn

`a.append(x)` thêm phần tử `x` vào **chính list a** (không tạo list mới, trả về `None`).
`a + b` tạo một list mới nối các phần tử, không sửa hai list ban đầu.
`b = a` chỉ tạo **bí danh** của a; muốn sao chép list phẳng dùng `a.copy()` hoặc `a[:]`.

Bẫy thường gặp: nếu lưu các hàng Pascal bằng cùng một list rồi sửa, các hàng cũ cũng đổi theo.
Với mảng giá cổ phiếu, hãy phân biệt rõ *thay đổi dữ liệu* với *đọc giá trong ngày*.
Thí nghiệm dưới đây sẽ chỉ ra sai lầm do aliasing một cách trực quan.

In [ ]:
original_list=[1,2,1]
alias_list=original_list
copy_list=original_list.copy()
original_list.append(1)
print('original:',original_list,'alias:',alias_list,'copy:',copy_list)
print('append trả:',copy_list.append(2))
print('copy đã thay đổi:',copy_list)
assert len(alias_list)==4 and len(copy_list)==4

### 2.4. `enumerate` cho cả vị trí và giá trị

Nếu cần biết phần tử đang ở **vị trí** nào (ngày thứ mấy, hàng thứ mấy),
`enumerate(items)` tạo lần lượt cặp `(index, value)`.
Nếu chỉ cần giá trị, `for value in items` gọn hơn.
`list(enumerate(...))` giúp quan sát trực tiếp các cặp chỉ số/giá trị.

Lưu ý `0-indexed`: ngày đầu hoặc hàng đầu có số thứ tự **0** trong Python.
Đây là nguyên nhân rất thường gặp của lỗi *off-by-one*: lệch đúng một vị trí.

In [ ]:
example_values=[7,1,5,3]
print('các cặp:',list(enumerate(example_values)))
for pos,value in enumerate(example_values):
    print(f'vị trí {pos} giữ giá trị {value}')
assert list(enumerate([8]))==[(0,8)]

### 2.5. Gọi hàm `min/max`, không nhầm với chuỗi phép so sánh

`min(a,b)` trả giá trị nhỏ hơn; `max(a,b)` trả giá trị lớn hơn.
Biểu thức `best=max(best,candidate)` là quy tắc phổ biến khi lưu **đáp án tốt nhất đến thời điểm hiện tại**.
Bạn có thể viết lại tương đương thành `if candidate > best: best = candidate`.

Hãy chú ý **best** giữ thông tin lịch sử, không nhất thiết bằng giá trị lớn nhất của chính input.
Ở bài Pascal, `max` không dùng để chọn số từ hàng cũ; số giữa được tính **bằng tổng**.
Cùng cú pháp Python, nhưng ý nghĩa toán học quyết định thuật toán.

In [ ]:
best_demo=0
for candidate_demo in (2,-3,5,4):
    old_best=best_demo
    best_demo=max(best_demo,candidate_demo)
    print('thử',candidate_demo,':',old_best,'->',best_demo)
assert best_demo==5

### 2.6. Định nghĩa Big-O bằng một ví dụ số nhỏ

**Time Complexity** nói số thao tác tăng thế nào khi input dài hơn.
Duyệt một vòng qua n phần tử thường tốn `O(n)`; hai vòng lồng nhau mỗi vòng chạy n lần
có thể tốn `O(n²)`. Tuy vậy phải đếm **số lần chạy thực** chứ không chỉ nhìn số vòng for:
nếu vòng trong chạy độ dài 1,2,3,...,n thì tổng là `n(n+1)/2 = O(n²)`.

**Space Complexity** tính dữ liệu phụ; nếu đầu ra yêu cầu chứa n phần tử thì riêng output đã cần `O(n)`.
Trong Pascal I trả toàn bộ tam giác cần `O(n²)` cho chính output, Pascal II chỉ trả một hàng `O(n)`.
Mục tiêu học là biết *n* đại diện cho cái gì trong từng bài.

In [ ]:
for n in (3,5,10):
    triangle_cells=n*(n+1)//2
    print(f'n={n}: số ô của n hàng đầu = {triangle_cells}; n² = {n*n}')
assert sum(range(1,6))==15

### 2.7. Quan sát trạng thái trước và sau khi cập nhật

Muốn hiểu một thuật toán theo vòng lặp, cần trả lời ba câu:
(1) Trước vòng lặp ta đã biết gì? (2) Ở bước này thay đổi biến nào?
(3) Sau bước này thông tin nào tiếp tục đúng?
Ta gọi thông tin được giữ đúng ấy là **invariant** (bất biến vòng lặp).

Để tránh bị rối, in `old_state` **trước** lệnh sửa và `new_state` **sau** lệnh sửa.
Nếu in mỗi đáp án cuối, ta không tìm được nguyên nhân lỗi ở giữa vòng lặp.
Bài minh họa dưới chỉ kiểm tra thao tác giữ tổng chạy, chưa sử dụng thuật toán của bài chính.

In [ ]:
running_sum=0
for number in [2,4,-1]:
    before=running_sum
    running_sum += number
    print(f'đọc {number:>2}: trước={before:>2}; sau={running_sum:>2}')
assert running_sum==5

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Cập nhật list **tại chỗ** và chiều duyệt. Khi đổi `row[j] += row[j-1]`, nếu duyệt từ trái sang phải sẽ vô tình dùng giá trị **vừa cập nhật**.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
row=[1,2,1]
wrong=row+[1]
for j in range(1,len(wrong)-1):wrong[j]+=wrong[j-1]
right=row+[1]
for j in range(len(right)-2,0,-1):right[j]+=right[j-1]
print('trái->phải (sai):',wrong)
print('phải->trái (đúng):',right)

### 3.1. Bài #119 khác #118 ở yêu cầu trả về

Bài #118 trả **tất cả hàng**, nên output có kích thước O(n²).
Bài #119 chỉ trả **một hàng** có chỉ số `rowIndex` (đếm từ 0).
Ví dụ `rowIndex=0 → [1]`, `rowIndex=3 → [1,3,3,1]`.
Ta có thể dựng toàn bộ từ hàng 0 đến rowIndex, nhưng giữ các hàng cũ là công việc không cần thiết
nếu đầu ra chỉ cần hàng cuối.

Bước tư duy tối ưu bộ nhớ: tại mỗi vòng lặp, giữ đúng **một** list hiện hành,
biến đổi nó thành hàng kế tiếp, không lưu lịch sử.
Tuy nhiên, khi biến đổi một list tại chỗ, phải xác định **chiều cập nhật an toàn**.

In [ ]:
expected119={0:[1],1:[1,1],2:[1,2,1],3:[1,3,3,1],4:[1,4,6,4,1]}
for index,row in expected119.items():
    print('rowIndex=',index,'trả:',row,'độ dài=',len(row))
    assert len(row)==index+1

### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Dùng lời giải #118 xây **toàn bộ** các hàng đến `rowIndex`, sau đó trả hàng cuối. Dễ đúng nhưng dùng O(k²) bộ nhớ.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(rowIndex):
    rows=[[1]]
    for i in range(1,rowIndex+1):
        prev=rows[-1];rows.append([1]+[prev[j-1]+prev[j] for j in range(1,len(prev))]+[1])
    return rows[-1]

In [ ]:
for k in [0,1,3,5]:print(k,'->',baseline(k))

### 4.1. Baseline lưu mọi hàng: dễ hiểu và đúng, nhưng dùng bộ nhớ nào?

Cách baseline giữ `rows=[[1]]`, sau đó tạo một hàng mới từ hàng trước rồi `append`.
Khi cần hàng k, cuối cùng trả `rows[-1]`. Giá trị đúng, nhưng các hàng 0..k−1
vẫn còn lưu trong `rows` dù không được trả về.
Tổng số ô lưu là `1+2+...+(k+1)=(k+1)(k+2)/2 = O(k²)`.

Cần phân biệt **một hàng cuối dài k+1** với **toàn bộ các hàng tổng k² ô**.
Mục tiêu tối ưu #119 là giữ bộ nhớ phụ O(k), tương đương kích thước hàng trả về.
Đây là một bước tối ưu *không đổi công thức toán học*, chỉ đổi cách quản lý dữ liệu.

In [ ]:
for idx in (0,3,6):
    total_cells=sum(range(1,idx+2))
    output_cells=idx+1
    print(f'rowIndex={idx}: baseline chứa {total_cells} ô; output cần {output_cells} ô')
    assert len(baseline(idx))==idx+1

### 4.2. Tại sao cập nhật list từ trái sang phải làm sai?

Hàng cũ `[1,2,1]` cần thành `[1,3,3,1]`.
Nếu thêm một số 1 ở cuối, ta có `[1,2,1,1]`.
Cập nhật **j=1** trước thì `row[1]=row[1]+row[0]=3`.
Đến **j=2**, công thức đọc `row[2]+row[1]=1+3=4`;
nhưng `row[1]` đã là **giá trị mới 3**, không còn giá trị cũ 2.
Kết quả sai `[1,3,4,1]`.

Chìa khóa là dữ liệu **chưa được phép ghi đè trước lúc dùng**.
Đi từ **phải sang trái** sẽ dùng `row[j-1]` khi nó vẫn giữ giá trị của hàng cũ.
Đây là một pattern tối ưu DP 2D xuống 1D rất quan trọng.

In [ ]:
wrong119=[1,2,1,1]
for j in range(1,3):
    old=wrong119[j]
    wrong119[j]+=wrong119[j-1]
    print('đi trái→phải j',j,'trước',old,'sau',wrong119)
assert wrong119==[1,3,4,1]
print('đúng phải là:',[1,3,3,1])

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Chỉ cần nhớ hàng **hiện tại**, không cần giữ các hàng cũ. Khi cập nhật **từ phải về trái**, ô `row[j-1]` vẫn còn giá trị cũ.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Bắt đầu `[1]`. Với mỗi hàng mới: thêm `1` ở cuối, rồi sửa các ô trong từ phải sang trái, `row[j] += row[j-1]`.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Chứng minh chiều cập nhật phải → trái bằng trạng thái cũ

Tại hàng k, mỗi ô giữa mới thỏa `new[j]=old[j-1]+old[j]`.
Giả sử đang cập nhật theo thứ tự j=k−1, k−2, ..., 1.
Khi xử lý ô j, các ô **bên phải** j có thể đã đổi, nhưng công thức chỉ đọc
`row[j]` (ô hiện tại chưa cập nhật) và `row[j-1]` (ô bên trái cũng chưa cập nhật).
Vì thế hai giá trị đọc được đều còn thuộc **hàng cũ**.

Đây là bất biến: trước khi sửa j, các vị trí 0..j vẫn giữ giá trị của hàng cũ.
Sau khi sửa j, vị trí j đúng giá trị mới; ta tiếp tục sang j−1.
Cuối cùng mọi ô giữa đúng, mép trái vẫn 1 và mép phải vừa append 1.

In [ ]:
reverse_update119=[1,2,1]
reverse_update119.append(1)
for j in range(len(reverse_update119)-2,0,-1):
    left_old=reverse_update119[j-1]
    this_old=reverse_update119[j]
    reverse_update119[j]=this_old+left_old
    print('j=',j,'dùng hai số cũ',this_old,left_old,'=>',reverse_update119)
assert reverse_update119==[1,3,3,1]

In [ ]:
def solve(rowIndex: int) -> list[int]:
    row=[1]
    for k in range(1,rowIndex+1):
        row.append(1)
        for j in range(k-1,0,-1):
            row[j]+=row[j-1]
    return row

### 6.1. Vòng ngoài và vòng trong của code O(k²)

`row=[1]` biểu diễn hàng 0.
Với `k` chạy `1..rowIndex`, ta chuyển hàng k−1 thành hàng k:
(1) `row.append(1)` thêm mép phải của hàng mới;
(2) `for j in range(k-1,0,-1)` đi từ ô giữa phải nhất về ô giữa trái nhất;
(3) `row[j] += row[j-1]` cộng hai phần tử cũ.

Ví dụ k=1: list trước `[1]`, thêm 1 thành `[1,1]`, vòng trong rỗng;
k=2: list trước `[1,1]`, thêm 1 thành `[1,1,1]`, sửa j=1 thành `[1,2,1]`.
Điều kiện stop=0 **không đưa j=0 vào loop**, nên mép trái luôn giữ 1.

In [ ]:
for k in range(1,5):
    print('k',k,'j sẽ được chạy:',list(range(k-1,0,-1)))
assert list(range(0,0,-1))==[]
assert list(range(1,0,-1))==[1]
assert list(range(3,0,-1))==[3,2,1]

### 6.2. Đặt mục tiêu kiểm thử: hàng 0, 1, 2 không được bỏ qua

Lỗi *off-by-one* thường bị che khuất khi chỉ thử rowIndex=4.
Ta nên kiểm tra từ nhỏ nhất:
- rowIndex=0: vòng ngoài không chạy, trả `[1]`;
- rowIndex=1: thêm 1 nhưng không có ô giữa, trả `[1,1]`;
- rowIndex=2: bắt đầu có một ô giữa, trả `[1,2,1]`;
- rowIndex=3: có hai ô giữa, trả `[1,3,3,1]`.

Nếu bạn viết `range(1,rowIndex)` thay vì `range(1,rowIndex+1)`,
chương trình sẽ thiếu chính hàng được yêu cầu. Nếu bạn viết `row.append(1)` sau cập nhật,
khi truy cập phần tử cũ có thể bị thiếu biên. Thứ tự thao tác là một phần của chứng minh đúng.

In [ ]:
for target in range(5):
    actual=solve(target)
    print(f'{target} -> {actual}')
    assert actual==expected119[target]

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Chuyển sang hàng | row trước | cập nhật từ phải | row sau |
|---:|---|---|---|
| 1 | `[1]` | không có ô giữa | `[1,1]` |
| 2 | `[1,1]` | `row[1]=1+1` | `[1,2,1]` |
| 3 | `[1,2,1]` | `row[2]=1+2`, rồi `row[1]=2+1` | `[1,3,3,1]` |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

### 7.1. Trace tay k=4: thứ tự j=3→2→1

Bắt đầu từ hàng 3: `[1,3,3,1]`. Thêm mép phải: `[1,3,3,1,1]`.

| Thứ tự | j | Trước `row[j]` | Trước `row[j-1]` | `row[j]` mới | Cả list sau bước |
|---:|---:|---:|---:|---:|---|
| 1 | 3 | 1 | 3 | 4 | `[1,3,3,4,1]` |
| 2 | 2 | 3 | 3 | 6 | `[1,3,6,4,1]` |
| 3 | 1 | 3 | 1 | 4 | `[1,4,6,4,1]` |

Hàng hoàn chỉnh có 5 phần tử, đối xứng và tổng bằng 16=2⁴.
Nếu chỉ đổi thứ tự sang j=1→3, ô thứ 3 sẽ dùng lại giá trị ô thứ 2 đã bị sửa,
vi phạm công thức “hai ô **hàng trước**”.

In [ ]:
row_trace119=[1,3,3,1]
row_trace119.append(1)
for j in range(3,0,-1):
    left_before=row_trace119[j-1]
    current_before=row_trace119[j]
    row_trace119[j]+=row_trace119[j-1]
    print(f'j={j}: {current_before}+{left_before}={row_trace119[j]}; list={row_trace119}')
assert row_trace119==[1,4,6,4,1]

### 7.2. Trace đầy đủ hai vòng để hiểu tổng số phép cộng

Khi rowIndex=k, số phép cộng ô giữa ở mỗi lượt hàng i là `max(0,i−1)`.
Tổng từ i=1 tới k là `0+1+...+(k−1)=k(k−1)/2`, tức O(k²).
**Không phải O(k)** chỉ vì ta chỉ lưu một list: độ phức tạp thời gian và bộ nhớ là hai câu hỏi khác nhau.

Trên bảng trace, mỗi dòng có thể gồm: hàng đang tạo, chỉ số j, giá trị hai ô cũ,
ô mới sau phép cộng và cả list hiện hành.
Với k=4, số phép cộng là 0+1+2+3=6. Hãy kiểm chứng bằng bộ đếm thay vì tin công thức.

In [ ]:
def count_additions119(k):
    row=[1];operations=0
    for i in range(1,k+1):
        row.append(1)
        for j in range(i-1,0,-1):
            row[j]+=row[j-1]
            operations+=1
    return row,operations
for k in (0,1,2,4,9):
    row,ops=count_additions119(k)
    print('k=',k,'phép cộng=',ops,'k*(k-1)//2=',k*(k-1)//2)
    assert row==solve(k) and ops==k*(k-1)//2

In [ ]:
def trace_row(k):
    row=[1]
    for i in range(1,k+1):
        print('trước hàng',i,':',row)
        row.append(1)
        for j in range(i-1,0,-1):
            before=row[j]
            row[j]+=row[j-1]
            print(f'  j={j}: {before}+{row[j-1]}->{row[j]}')
        print('sau:',row)
    return row
trace_row(4)

### 7.3. Cách thứ ba: công thức hệ số tổ hợp O(k) bước

Khi biết hàng k có các hệ số `C(k,0), C(k,1), ..., C(k,k)`, có thể sinh lần lượt:
`C(k,0)=1`, `C(k,j+1)=C(k,j) * (k-j) // (j+1)`.
Ví dụ k=4: hệ số 1 → 4 → 6 → 4 → 1.
Công thức xuất phát từ cách rút gọn phân số giai thừa:
`C(k,j+1) / C(k,j) = (k-j)/(j+1)`.

**Lưu ý** công thức lặp này có O(k) bước số học, còn phương án DP in-place là O(k²) phép cộng.
Bài nhập môn học DP trước vì nó dễ mở rộng sang các bài có *không có* công thức tổ hợp.
Khi phân tích độ phức tạp bit đối với số nguyên rất lớn, phép nhân/chia không phải O(1);
trong phạm vi LeetCode #119 nhỏ, người ta thường đếm phép toán số học.

In [ ]:
def pascal_by_binomial119(k):
    ans=[];c=1
    for j in range(k+1):
        ans.append(c)
        if j<k:c=c*(k-j)//(j+1)
    return ans
for k in (0,1,2,4,10,33):
    assert pascal_by_binomial119(k)==solve(k)
print('PASS công thức tổ hợp bằng in-place DP')

### 7.4. Vẽ hàng theo đối xứng để tự kiểm tra kết quả

Hàng Pascal luôn đối xứng, vì chọn j phần tử từ k phần tử tương đương chọn k−j phần tử **không chọn**.
Giá trị lớn nhất thường ở giữa: nếu k chẵn, một hệ số chính giữa;
nếu k lẻ, hai hệ số giữa bằng nhau.

Đối xứng và tổng bằng `2**k` là hai thuộc tính kiểm thử độc lập,
nhưng một hàng giả vẫn có thể thỏa một thuộc tính riêng lẻ,
nên tốt nhất so cả `solve(k)` với baseline hoặc công thức tổ hợp.
Đoạn code kiểm tra cả độ dài, hai đầu, đối xứng và tổng cho hàng k=0..14.

In [ ]:
for k in range(15):
    row=solve(k)
    assert len(row)==k+1
    assert row[0]==row[-1]==1
    assert row==row[::-1]
    assert sum(row)==2**k
    print('k=',k,'độ dài',len(row),'đối xứng',True)

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Trước khi bắt đầu hàng mới, `row` là hàng trước đúng. Duyệt từ phải sang trái đảm bảo `row[j]` và `row[j-1]` vẫn là dữ liệu **của hàng trước**, nên phép cộng tạo đúng ô ở hàng mới. Hai đầu vẫn bằng 1. Quy nạp theo từng hàng.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
class Solution:
    def getRow(self, rowIndex: int):
        row=[1]
        for k in range(1,rowIndex+1):
            row.append(1)
            for j in range(k-1,0,-1):
                row[j]+=row[j-1]
        return row

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for k in [0,1,2,3,5,10,33]:
    assert solve(k)==baseline(k)
    assert Solution().getRow(k)==baseline(k)
print('PASS Pascal II, có k=33')

### 10.1. Tất cả trường hợp k=0..33 và đối chiếu cả ba lời giải

Các test gốc chỉ có vài giá trị k. Để đảm bảo đúng ranh giới, hãy chạy toàn bộ
k từ 0 đến 33 — phạm vi thường gặp trong đề #119.
So `solve` (một list), `baseline` (mọi hàng), công thức tổ hợp (từng hệ số),
và code submit `Solution().getRow`.

Ba cách dùng cùng định nghĩa toán học nhưng triển khai khác nhau:
so chéo giúp phát hiện lỗi cập nhật chỉ số, sai bước cộng, hoặc lỗi hàng đầu.
Với k=33, số nguyên lớn nhất vẫn chính xác trong Python (không dùng float).
Đừng kiểm tra chỉ `row[-1]`: mép phải luôn bằng 1 kể cả khi các ô trong bị sai.

In [ ]:
for k in range(34):
    expectation=pascal_by_binomial119(k)
    assert solve(k)==baseline(k)==Solution().getRow(k)==expectation
print('PASS 34 hàng (index 0..33) với ba phương pháp độc lập')

### 10.2. Test sự độc lập: gọi hàm nhiều lần không giữ trạng thái cũ

Hàm chuẩn nhận rowIndex, tạo `row` mới mỗi lần và trả kết quả.
Không nên dùng list toàn cục để lưu hàng giữa các lần gọi, vì nếu lần sau
rowIndex nhỏ hơn hoặc lớn hơn, kết quả có thể bị ảnh hưởng từ lượt trước.

Thử gọi xen kẽ `getRow(10)`, `getRow(0)`, `getRow(5)` rồi lại `getRow(10)`;
hai kết quả hàng 10 phải giống nhau. Nếu chỉnh sửa list của lần gọi trước,
những lần gọi mới vẫn phải hoạt động vì không tái sử dụng list cũ.
Đây là cách kiểm tra lỗi **hidden mutable state** thường gặp trong notebook.

In [ ]:
first119=solve(10)
assert solve(0)==[1]
assert solve(5)==[1,5,10,10,5,1]
first119[0]=999
assert solve(10)[0]==1
assert Solution().getRow(10)==baseline(10)
print('PASS nhiều lần gọi độc lập nhau')

## 11. Phân tích Time/Space Complexity

Đặt `k=rowIndex`. **Time O(k²)** vì tính tổng 1+...+k; **Space O(k)** vì lưu chỉ một hàng. Baseline O(k²) bộ nhớ khi giữ toàn bộ tam giác.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Duyệt từ trái sang phải làm dính giá trị vừa cập nhật.
- ❌ Cho rằng `rowIndex=1` trả `[1]` vì quên đánh số từ 0.
- ❌ Đổi chiều mà không quan sát dữ liệu nào còn cũ.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Sai lầm thực tế: dùng `range(1,k)` và bỏ mất hàng cuối

Nếu cần sinh hàng k, vòng ngoài phải chạy các bước 1,2,...,k.
`range(1,k)` dừng ở k−1, vì stop bị loại trừ; như thế thuật toán trả hàng k−1.
Ví dụ yêu cầu hàng 4 nhưng vòng lặp chỉ chạy 1,2,3,
nên kết quả sai `[1,3,3,1]` thay vì `[1,4,6,4,1]`.

Đây là lỗi *off-by-one* rất phổ biến trong các bài mảng, binary search và DP.
Cách chống lỗi: trước khi viết code, xác định **số bước cập nhật** từ base case đến target;
từ hàng 0 đến hàng k cần đúng k lần chuyển, không phải k−1.

In [ ]:
def wrong_stop119(k):
    row=[1]
    for i in range(1,k):  # SAI: không làm bước k
        row.append(1)
        for j in range(i-1,0,-1):row[j]+=row[j-1]
    return row
print('sai k=4:',wrong_stop119(4),'đúng:',solve(4))
assert wrong_stop119(4)!=solve(4)

### 12.2. Sai lầm: `row[j] = row[j] + row[j+1]` thay vì ô bên trái

Công thức đúng ở hàng n là **tổng hai ô của hàng cũ** tại vị trí j−1 và j.
Khi cập nhật từ phải về trái, ta phải dùng `row[j-1]`, không phải `row[j+1]`.
Nếu lấy ô bên phải j+1, ô đó đã có thể được cập nhật ở lượt trước
và còn vượt biên khi j ở gần cuối.

Bảng tam giác có sơ đồ cụ thể: hai ô phía trên của cột j nằm ở cột **j−1 và j**,
không phải j và j+1. Vẽ 3 hàng trước khi viết index là cách tránh nhầm hiệu quả.
Thí nghiệm dưới chỉ tính ra hai phép cộng để so, không chạy code cố tình truy cập ngoài list.

In [ ]:
old119=[1,3,3,1]
j119=2
correct_new=old119[j119-1]+old119[j119]
wrong_right=old119[j119]+old119[j119+1]
print('ô giữa cần giá trị:',correct_new,'nếu lấy nhầm ô phải:',wrong_right)
assert correct_new==6 and wrong_right==4

## 13. Nhận diện dạng bài và vận dụng

Cần tối ưu DP từ **2D xuống 1D**: xác định hướng cập nhật an toàn để không ghi đè thông tin còn cần dùng.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tự xây tất cả hàng bằng cách gọi `solve(k)` cho `k=0..n-1`, rồi đối chiếu với #118.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
print('k=0..4:',[solve(k) for k in range(5)])

### 14.1. Ứng dụng khác của cập nhật 1D từ phải sang trái

Quy tắc quan trọng không riêng Pascal: nếu trạng thái mới của vị trí j
cần **trạng thái cũ của j−1**, thì cập nhật từ trái sang phải dễ gây ghi đè.
Một trường hợp tương tự là bài **0/1 knapsack** (mỗi món chọn tối đa một lần):
DP một chiều phải quét dung lượng từ phải sang trái để không dùng cùng món nhiều lần.

Không nên máy móc luôn quét ngược: có bài *unbounded knapsack* phải quét xuôi
vì cho phép dùng một món nhiều lần. Cần hỏi “giá trị bên trái ở vòng này đã được phép là mới chưa?”.
Ở Pascal II, câu trả lời là **chưa**, nên cần đi ngược.

In [ ]:
# Mô phỏng chính xác một bước 0/1-knapsack với một món trọng lượng 2, giá trị 5.
dp119=[0]*6
for capacity in range(5,1,-1):
    dp119[capacity]=max(dp119[capacity],dp119[capacity-2]+5)
print('0/1 item chỉ dùng một lần:',dp119)
assert dp119==[0,0,5,5,5,5]

### 14.2. Quan hệ giữa hàng Pascal và hệ số khai triển nhị thức

Theo **nhị thức Newton**, các hệ số của `(a+b)^n` là các số ở hàng n của tam giác Pascal.
Ví dụ `(a+b)^3 = a³ + 3a²b + 3ab² + b³` nên hàng 3 là `[1,3,3,1]`.
Dùng a=b=1 ta suy ra tổng hệ số hàng n bằng `2^n`.

Đây là cách kiểm chứng ý nghĩa toán học của Pascal và một ví dụ về cách một thuật toán
lưu các hệ số cho bài toán khai triển biểu thức. LeetCode không yêu cầu in đa thức;
ta chỉ in hàng số nguyên. Vận dụng kiến thức bên ngoài được tách riêng để không làm rối code submit.

In [ ]:
for n in range(7):
    coeff=solve(n)
    value_at_1_1=sum(coeff)
    print('n=',n,'hệ số=',coeff,'(1+1)^n=',value_at_1_1)
    assert value_at_1_1==2**n

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Với `row=[1,3,3,1]`, trước khi cập nhật, đoán `row[2] + row[1]` và giải thích vì sao phải sửa `row[2]` **trước** `row[1]`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [ ]:
row=[1,3,3,1]
print("giá trị mới của row[2] =",row[2]+row[1])
print("giá trị mới của row[1] =",row[1]+row[0])

### Bài 2 — Trace bằng tay (hiểu bản chất)

Từ `[1,2,1]`, dựng hàng 4 phần tử, rồi hàng 5 phần tử **in-place**. Viết rõ thứ tự cập nhật các chỉ số j.

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_get_pascal_row(rowIndex)` chỉ lưu một list hàng hiện tại và cập nhật **từ phải sang trái**.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [ ]:
def exercise_get_pascal_row(rowIndex):
    # TODO: bắt đầu [1], thêm 1 rồi cập nhật các ô giữa theo chiều ngược.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Viết `exercise_get_pascal_row_copy(rowIndex)` dựng từng hàng bằng `prev` → `new`, không ghi đè in-place. So sánh cách này với bài 3.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [ ]:
def exercise_get_pascal_row_copy(rowIndex):
    # TODO: prev bắt đầu [1]; mỗi vòng tạo new=[1,...,1].
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: tính hàng bằng công thức C(n,k) trong `exercise_get_pascal_row_formula(rowIndex)`, chỉ dùng O(n) thời gian thay vì O(n²).

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [ ]:
def exercise_get_pascal_row_formula(rowIndex):
    # TODO: hệ số đầu tiên là 1; hệ số sau = c*(n-k)//(k+1).
    raise NotImplementedError

### Bài 6 — TODO: tính hàng Pascal **theo modulo**

Cho số nguyên `mod > 0`, viết `exercise_pascal_row_mod(rowIndex,mod)` trả
mỗi phần tử của hàng Pascal theo phép chia lấy dư `mod`.
Ví dụ hàng 4 là `[1,4,6,4,1]`; với mod=2 trả `[1,0,0,0,1]`.

Đây không chỉ là gọi `solve(rowIndex)` rồi `%` toàn bộ output:
hãy **tự viết vòng cập nhật phải→trái** và áp dụng `% mod` ngay tại thời điểm cộng,
để luyện khái niệm trạng thái cũ và hạn chế kích thước số nguyên trung gian.
Gợi ý 1: bắt đầu `[1%mod]`; mỗi vòng append `1%mod`.
Gợi ý 2: `row[j]=(row[j]+row[j-1])%mod` từ j lớn về j nhỏ.
Gợi ý 3: với mod=1 mọi phần tử sẽ bằng 0, kể cả hai mép.

In [ ]:
def exercise_pascal_row_mod(rowIndex,mod):
    # TODO: dùng DP in-place từ phải sang trái, thực hiện modulo ngay mỗi bước.
    raise NotImplementedError

# Sau khi tự làm:
# assert exercise_pascal_row_mod(4,2)==[1,0,0,0,1]
# assert exercise_pascal_row_mod(2,1)==[0,0,0]

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Lỗi phổ biến: cập nhật từ trái qua phải làm `row[j-1]` **đã bị đổi**.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Trong vòng ngoài k, `row.append(1)` rồi `for j in range(k-1,0,-1)` để sửa.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

C(n,0)=1, C(n,k+1)=C(n,k)*(n-k)//(k+1). Dãy hệ số tạo ra hàng Pascal.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Kết quả lần lượt `6` và `4`. Cập nhật từ **phải về trái** để không ghi đè phần tử trái trước khi dùng nó.

### Bài 2 — Trace tay

Thêm 1 vào cuối: `[1,2,1,1]`, cập nhật `j=2` được `[1,2,3,1]`, `j=1` được `[1,3,3,1]`. Hàng tiếp theo: thêm 1, rồi `j=3,2,1` để ra `[1,4,6,4,1]`.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [ ]:
def answer_get_pascal_row(rowIndex):
    row=[1]
    for k in range(1,rowIndex+1):
        row.append(1)
        for j in range(k-1,0,-1): row[j]+=row[j-1]
    return row

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [ ]:
def answer_get_pascal_row_copy(rowIndex):
    prev=[1]
    for k in range(1,rowIndex+1):
        prev=[1]+[prev[j-1]+prev[j] for j in range(1,k)]+[1]
    return prev

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [ ]:
def answer_get_pascal_row_formula(rowIndex):
    row=[];c=1
    for k in range(rowIndex+1):
        row.append(c)
        if k<rowIndex: c=c*(rowIndex-k)//(k+1)
    return row

### Đáp án bài 6 — một chiều, cập nhật từ phải sang trái, lấy dư từng bước

Lời giải giữ một list duy nhất. Các ô biên 1 phải được lấy `%mod`,
vì với mod=1, 1 có dư 0. Các ô bên trong dùng quy tắc cộng Pascal rồi lấy dư.
Tính chất `(a+b)%m == ((a%m)+(b%m))%m` bảo đảm giảm modulo mỗi bước không đổi kết quả cuối.

Bài này minh họa cách giữ trạng thái số học nhỏ gọn,
áp dụng cho các bài đếm DP có câu “trả đáp án modulo 10^9+7”.
Nếu đề có `mod=0` thì phép chia lấy dư không hợp lệ;
ở bài tập đã quy định rõ `mod>0`, nên không cần tự chế quy tắc cho input sai.

In [ ]:
def answer_pascal_row_mod(rowIndex,mod):
    row=[1%mod]
    for i in range(1,rowIndex+1):
        row.append(1%mod)
        for j in range(i-1,0,-1):
            row[j]=(row[j]+row[j-1])%mod
    return row
for k in range(0,19):
    for modulus in (1,2,3,5,97):
        assert answer_pascal_row_mod(k,modulus)==[x%modulus for x in solve(k)]
print('PASS bài 6: modulo ngay trong bước cập nhật')

### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [ ]:
for k in [0,1,2,3,4,5,10,20]:
    expected=answer_get_pascal_row(k)
    assert answer_get_pascal_row_copy(k)==expected
    assert answer_get_pascal_row_formula(k)==expected
assert answer_get_pascal_row(4)==[1,4,6,4,1]
print("PASS: đáp án 3 bài coding của phần luyện tập")

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Chỉ cần nhớ hàng **hiện tại**, không cần giữ các hàng cũ. Khi cập nhật **từ phải về trái**, ô `row[j-1]` vẫn còn giá trị cũ.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.